In [1]:
# install required dependencies
%pip install psycopg2-binary
%pip install dash
%pip install dash-bootstrap-components

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# imports
import psycopg2
from psycopg2 import sql
from configparser import ConfigParser
from dash import Dash, html, dcc, Input, Output, State, dash_table
import dash_bootstrap_components as dbc
from datetime import date
import threading
from IPython.display import IFrame, display

In [3]:
# CONFIGURATION

# Read data from a configuration file and store in a dictionary
# as key-value pairs.
#
# Parameters:
#   filename: The name of the file that contains the configuration data.
#   section:  The name of the section within the configuration file
#             for which we are retrieving data.
# Returns: A dictionary with the key-value pairs from the configuration file.
def config(filename, section):
    cp = ConfigParser()
    # acquire data
    cp.read(filename)
    
    # convert to dictionary
    block = {}
    # make sure each section exists
    if cp.has_section(section):
        # read the entire section
        items = cp.items(section)
        # loop through config items to create the dictionary
        for item in items:
            block[item[0]] = item[1]
    else:
        # a section is missing, error out
        raise Exception(f"Section {section} was not found in the {filename} file!")
    return block

# Parse the provided config file as display results. Useful for testing.
#
# Parameters:
#   filename: The name of the file that contains the configuration data.
#   section:  The name of the section within the configuration file
#             for which we are retrieving data.
def displayConfig(filename, section):
    # obtain the config
    confData = config(filename, section)
    # display
    print(confData)

In [4]:
# DATABASE CONNECTION

# Connect to a specific PostgreSQL database leveraging the config() method.
#
# Parameters:
#   filename: The name of the file that contains the configuration data.
#   section:  The name of the section within the configuration file
#             for which we are retrieving data.
# 
# Returns: A Connection to the databse or null in the event of a problem.
def dbConnect(filename, section):
    # obtain config
    confData = config(filename, section)

    conn = None
    try:     
        # attempt to connect
        conn = psycopg2.connect(
            dbname = confData["database"],
            # NOTE: I've added the port as a configuration parameter because my
            # machine uses 5433 instead of the default 5432
            port = confData["port"],
            user = confData["user"],
            password = confData["password"],
            host = confData["host"]
        )
        # return the connection for use
        # don't forget to close the connection when done!
        return conn
    except(Exception, psycopg2.DatabaseError) as error:
        # failed to connect, show the issue
        print(f"ERROR: Unable to connect to PostgreSQL database: {error}")
        return None
    
# Test the connection to the database using the provided config file.
# Connect, validate the current database, and close the connection.
# Useful for testing.
# 
# Parameters:
#   filename: The name of the file that contains the configuration data.
#   section:  The name of the section within the configuration file
#             for which we are retrieving data.
def dbTest(filename, section):
    # create connection
    print('Connecting to our PostgreSQL database...')
    conn = dbConnect(filename, section)

    try:     
        # attempt to make a cursor and use it to execute a query
        cursor = conn.cursor()
        cursor.execute('SELECT current_database()')
        # display result
        print(cursor.fetchone())
        # close the cursor
        cursor.close()
    except(Exception, psycopg2.DatabaseError) as error:
        print(f"ERROR: Unable to create and use cursor: {error}")
    finally:
        # all good, close the connection
        if conn is not None:
            conn.close()

In [5]:
# CRUD METHODS
# Read one or more records from the specified table matching the given clause.
#
# Parameters:
#     tablename: The table to select the record(s) from.
#     columns: The list of columns to retrieve.
#     clause: The appropriate clause to test for - specifically what you would 
#             see after the SQL WHERE clause.
#     conn: The database connection to use to read the record(s).
# Returns: The number of retrieved rows or -1 on error.
def readRecord(tablename, columns, clause, conn):
    cursor = None
    try:     
        # create a cursor
        cursor = conn.cursor()

        # format the query with safe string composition
        query = sql.SQL("SELECT {cols} FROM public.{table} WHERE {clause}").format(
            cols=sql.SQL(',').join([sql.Identifier(x) for x in columns]),
            table=sql.Identifier(tablename),
            clause=sql.SQL(clause)
        )
        # execute the read operation
        print(f"Reading from {tablename}...")
        cursor.execute(query)
        rows = cursor.rowcount
        print(f"{rows} record(s) read successfully!")
        
        # return all results
        return cursor.fetchall()
    except(Exception, psycopg2.DatabaseError) as error:
        print("ERROR: Failed to read record(s):", error)
        return []
    finally:
        # close the cursor as we are done with this operation
        if cursor:
            cursor.close()
            

# Read one or more records from the specified table using an arbitrary query.
# This is so that joins can be used.
#
# Parameters:
#     query: The query to execute.
#     conn: The database connection to use read the record(s).
# Returns: The number of retrieved rows or -1 on error.
def readRecordArbitrary(query, conn):
    cursor = None
    try:     
        # create a cursor
        cursor = conn.cursor()

        # format the query with safe string composition
        query = sql.SQL(query)
        # execute the read operation
        print(f"Reading...")
        cursor.execute(query)
        rows = cursor.rowcount
        print(f"{rows} record(s) read successfully!")
        
        # return all results
        return cursor.fetchall()
    except(Exception, psycopg2.DatabaseError) as error:
        print("ERROR: Failed to read record(s):", error)
        return []
    finally:
        # close the cursor as we are done with this operation
        if cursor:
            cursor.close()

In [6]:
# define some global constants
CONFIG_PATH = 'database.conf'
CONFIG_SECTION = 'postgresql'

In [7]:
# the dashboard will always need to display the list of categories, so get them upfront
conn = dbConnect(CONFIG_PATH, CONFIG_SECTION)
categories = readRecord("ctgnme", ["Food category code", "Category description"], "TRUE", conn)

# get today's date and the earliest last modified date
today = date.today().strftime('%Y-%m-%d')
query = 'SELECT MIN(f."Last modified") FROM public.fdes as f'
min_date_result = readRecordArbitrary(query, conn)
if min_date_result and min_date_result[0][0]:
    min_date = str(min_date_result[0][0])[:10] # cuts off time if it's a timestamp
else:
    min_date = "1970-01-01" # safe fallback date if table is empty

# close the connection
conn.close()

Reading from ctgnme...
25 record(s) read successfully!
Reading...
1 record(s) read successfully!


In [8]:
# create the dash app with bootstrap theme
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# layout components
header = dbc.Row([
    dbc.Col(
        html.H2(
            "USDA Food Product Dashboard", 
            className="text-center my-4",
            style={'fontFamily': 'Arial, sans-serif', 'fontWeight': 'bold'}
        ),
        width=12 # number of grid columns
    )
])

category_search = dcc.Input(
    id='category-search',
    type='text',
    placeholder='search categories...',
    style={'width': '100%', 'fontSize': '0.8rem'},
    className='form-control mb-3'
)

# format category labels "1 | Dairy and Egg Products"
checklist_options = [
    {'label': f"{item_id} | {name}", 'value': item_id} 
    for item_id, name in categories
]

# scrollable container holding the checklist
category_checklist = html.Div([
    dcc.Checklist(
        id='category-checklist',
        options=checklist_options,
        value=[], # start with no categories checked by default
        labelStyle={'fontSize': '0.7rem', 'display': 'block', 'marginBottom': '10px'},
        inputStyle={'marginRight': '10px'}
    )
], style={
    'maxHeight': '450px', 
    'overflowY': 'auto', 
    'paddingRight': '5px',
    'flexGrow': '1',
    'marginTop': '10px'
})

categories_panel = dbc.Col([
    html.Div([
        html.H3("Food Category", style={'fontSize': '1.0rem', 'fontWeight': 'bold'}),
        category_search,
        category_checklist
    ], className="h-100", style={
        'border': '2px solid #333', 
        'borderRadius': '15px', 
        'padding': '15px', 
        'minHeight': '500px'
    }),
], width=4, className="d-flex flex-column") # take up 4 out of 12 grid columns

result_search = dbc.Col([
    dcc.Input(
        id='result-search',
        type='text',
        placeholder='search description...',
        className='form-control',
        style={'width': '100%', 'fontSize': '0.8rem'}
    )
], width=7, className="pe-0")

datepicker = dbc.Col([
    dcc.DatePickerRange(
        id='datepicker',
        start_date=min_date,
        end_date=today,
        min_date_allowed=date(1970, 1, 1), 
        max_date_allowed=today, 
        initial_visible_month=today, 
        style={'width': '100%', 'fontSize': '0.7rem'}
    )
], width=5, className="d-flex justify-content-end")

results_panel = dbc.Col([
    html.Div([
        dbc.Row([result_search, datepicker], className="mb-3 align-items-center"),
        html.Div(
            "Select a food category to get started!", 
            id="query-results-container",
            className="text-muted d-flex align-items-center justify-content-center",
            style={'height': '100%', 'minHeight': '460px'}
        )
    ], className="h-100", style={
        'border': '2px solid #333', 
        'borderRadius': '5px', 
        'padding': '15px', 
        'minHeight': '500px'
    })
], width=8, className="d-flex flex-column") # take up 8 out of 12 grid columns

# Create table of results.
#
# Parameters:
#     rows: The database rows of results (category, description, modified date)
# Returns: A formatted result table or a string indicating no products found and
# a string of css classes
def create_results_table(rows):
    if not rows:
        return "No products found!", "text-muted p-3 text-center"

    table_data = [
        {
            "category": row[0],
            "description": row[1],
            "last_modified": str(row[2])
        } for row in rows
    ]
    
    # use Dash's compiled DataTable component because it is much faster than an HTML table
    table = dash_table.DataTable(
        data=table_data,
        columns=[
            {"name": "Category", "id": "category"},
            {"name": "Description", "id": "description"},
            {"name": "Last Modified Date", "id": "last_modified"}
        ],
        # pagination for even more speed/less overload
        page_size=10,
        # add asc/desc sorting options
        sort_action="native",
        # make sure the arrows show the default ordering from the SQL query
        sort_by=[
            {"column_id": "category", "direction": "asc"},
            {"column_id": "description", "direction": "asc"}
        ],
        
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'padding': '10px',
            'fontFamily': 'Arial, sans-serif',
            'fontSize': '0.7rem'
        },
        style_header={
            'backgroundColor': '#f8f9fa',
            'fontWeight': 'bold',
            'border': '1px solid #dee2e6'
        },
        style_data={
            'border': '1px solid #dee2e6'
        }
    )
    return table, "p-1 text-dark text-start"

app.layout = dbc.Container([
    header,
    dbc.Row([categories_panel, results_panel]),
], fluid=True)

In [9]:
# INTERACTIVITY

# category search
# NOTE: I'm choosing to just filter in the application rather than the
# database because there is a relatively small number of categories
@app.callback(
    Output('category-checklist', 'options'),
    Input('category-search', 'value'),
    State('category-checklist', 'value') # reads checked boxes without triggering the callback itself
)
def filter_categories(search_value, selected_values):
    # ensure selected_values is a list to prevent syntax errors
    if selected_values is None:
        selected_values = []
        
    # add valid options to the list
    options = []
    for item_id, name in categories:
        label = {'label': f"{item_id} | {name}", 'value': item_id}
        
        # if the search bar is empty show everything
        if not search_value:
            options.append(label)
            continue
            
        # make sure selected categories are always shown
        if item_id in selected_values:
            options.append(label)
            continue
        
        # finally, add if it matches filter
        if search_value.lower() in name.lower():
            options.append(label)        
    return options

# any time something changes the results pane, eg.
#     a category is selected/deselected
#     product description search is updated
#     date range is changed
# go out to the database and fetch new results
@app.callback(
    Output('query-results-container', 'children'),
    Output('query-results-container', 'className'),
    Input('category-checklist', 'value'),
    Input('result-search', 'value'),
    Input('datepicker', 'start_date'),
    Input('datepicker', 'end_date')
)
def fetch_results(selected_category_ids, product_search_value, start_date, end_date):
    # no categories are selected, show placeholder call to action
    if not selected_category_ids:
        return "Select a food category to get started!", "text-muted d-flex align-items-center justify-content-center"
        
    # build query
    ids = ",".join(str(item_id) for item_id in selected_category_ids)
    
    # start with empty clause for when no filtering is needed
    description_clause = ""
    if product_search_value and product_search_value.strip() != "":
        # clean up any potential single quotes to prevent breaking the raw string
        safe_search = product_search_value.replace("'", "''").strip()
        # ILIKE makes the pattern match case-insensitive in PostgreSQL
        description_clause = f"AND f.\"Descriptor\" ILIKE '%{safe_search}%'"
        
    date_clause = ""
    if start_date and end_date:
        # standard ISO format date strings 'YYYY-MM-DD' map directly into SQL
        date_clause = f"AND f.\"Last modified\" BETWEEN '{start_date}' AND '{end_date}'"
    elif start_date:
        # filter start date and forward
        date_clause = f"AND f.\"Last modified\" >= '{start_date}'"
    elif end_date:
        # filter everything up to and including end date
        date_clause = f"AND f.\"Last modified\" <= '{end_date}'"
        
    query = f"""
        SELECT c."Category description", f."Descriptor", f."Last modified"
        FROM public.fdes as f JOIN public.ctgnme as c
        ON f."Food category code" = c."Food category code"
        WHERE f."Food category code" IN ({ids})
        {description_clause}
        {date_clause}
        ORDER BY c."Category description" ASC, f."Descriptor" ASC
    """
    # connect and execute
    conn = dbConnect(CONFIG_PATH, CONFIG_SECTION)
    rows = readRecordArbitrary(query, conn)
    # best practice to close connectionclose
    conn.close()
    
    #format
    table = create_results_table(rows)
    return table

In [10]:
# run the app!
# app.run(jupyter_mode="external", port=8050)

# run inline right inside jupyter notebook
app.run(jupyter_mode="inline", port=8050)